# Rice Disease Classification - YOLO11n-cls

Google Colab pipeline for the audit-approved RiceLeafDiseaseBD classification set. It uses pretrained yolo11n-cls.pt, 224 px input, at most 30 epochs, AdamW, early stopping, and best-checkpoint saving.

This notebook does not evaluate the test set. Training and checkpoint selection use only train.csv and val.csv. The comparison notebook evaluates the held-out test manifest after both model checkpoints exist.

Training and export are explicit opt-ins and default to False. Select a Colab GPU before training.

Preprocessing: Ultralytics classification training augmentation and its built-in resize/crop/tensor pipeline are retained at 224 px. This notebook does not apply a custom ImageNet mean/std transform. References: [Ultralytics classification](https://docs.ultralytics.com/tasks/classify), [YOLO11](https://docs.ultralytics.com/models/yolo11).

In [ ]:
%pip -q install ultralytics==8.4.215 pandas scikit-learn matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, platform, random
import numpy as np
import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO

SEED = 42
IMG_SIZE = 224
MAX_EPOCHS = 30
EARLY_STOP_PATIENCE = 8
BATCH_SIZE = 32
START_TRAINING = False  # Set True explicitly after reviewing the preflight.
RUN_EXPORTS = False     # Set True explicitly after training has completed.

DRIVE_ROOT = Path('/content/drive/MyDrive/Krishok_AI')
DATA_ROOT = DRIVE_ROOT / 'data/processed/rice_classification'
MANIFEST_DIR = DATA_ROOT / 'manifests'
RUNS_ROOT = DRIVE_ROOT / 'runs/rice_classification'
YOLO_RUN_DIR = RUNS_ROOT / 'yolo11n_cls'
TRAIN_CSV = MANIFEST_DIR / 'train.csv'
VAL_CSV = MANIFEST_DIR / 'val.csv'
CLASS_MAP_PATH = MANIFEST_DIR / 'class_mapping.json'
CROSS_REVIEW_PATH = MANIFEST_DIR / 'cross_class_review.csv'
EXPECTED_COUNTS = {'train': 6827, 'val': 1463}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print({'python': platform.python_version(), 'torch': torch.__version__,
       'ultralytics': ultralytics.__version__, 'cuda': torch.version.cuda,
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab before enabling training.'

## Read-only preflight

This checks only train and validation. It does not read test.csv or test images. It compares CSV paths with the folder inventory consumed by Ultralytics and checks duplicate-group separation.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

class_map = json.loads(CLASS_MAP_PATH.read_text(encoding='utf-8'))
classes = class_map['classes']
class_to_id = {item['class_label']: int(item['class_id']) for item in classes}
assert class_map['uses_yolo_annotation_class_ids'] is False
assert sorted(class_to_id.values()) == list(range(len(class_to_id)))
approved_class_order = [label for label, idx in sorted(class_to_id.items(), key=lambda item: item[1])]

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
assert len(train_df) == EXPECTED_COUNTS['train'] and len(val_df) == EXPECTED_COUNTS['val']
assert set(train_df.class_label) == set(class_to_id) == set(val_df.class_label)

for split, frame in [('train', train_df), ('val', val_df)]:
    folder_class_order = sorted(p.name for p in (DATA_ROOT / split).iterdir() if p.is_dir())
    assert folder_class_order == approved_class_order, f'Ultralytics folder classes differ from approved mapping in {split}: {folder_class_order}'
    assert frame['image_path'].is_unique, f'Duplicate manifest path in {split}'
    expected_ids = frame['class_label'].map(class_to_id).astype(int)
    assert frame['class_id'].astype(int).equals(expected_ids), f'Class IDs differ in {split}'
    missing = [p for p in frame.image_path if not (DATA_ROOT / p).is_file()]
    assert not missing, f'{len(missing)} missing {split} images; examples: {missing[:3]}'
    inventory = {p.relative_to(DATA_ROOT).as_posix()
                 for c in class_to_id
                 for p in (DATA_ROOT / split / c).rglob('*')
                 if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}}
    assert inventory == set(frame.image_path), f'{split} folders differ from the approved CSV manifest'

train_groups, val_groups = set(train_df.duplicate_group_id), set(val_df.duplicate_group_id)
assert not (train_groups & val_groups), 'An MD5 group crosses train and validation.'
review = pd.read_csv(CROSS_REVIEW_PATH)
unresolved = set(review.loc[review.review_status == 'unresolved_excluded', 'group_id'])
assert not (unresolved & (train_groups | val_groups)), 'An unresolved duplicate group is included.'
print({'class_mapping': class_to_id, 'train': len(train_df), 'val': len(val_df),
       'train_manifest_sha256': sha256_file(TRAIN_CSV), 'val_manifest_sha256': sha256_file(VAL_CSV),
       'unresolved_cross_class_groups_excluded': len(unresolved)})

## Training (explicit opt-in)

Ultralytics validates on val during training for early stopping and best-checkpoint selection. The test split is not passed to this call.

In [ ]:
if START_TRAINING:
    if YOLO_RUN_DIR.exists():
        raise FileExistsError(f'Run directory already exists; choose a new path: {YOLO_RUN_DIR}')
    YOLO_RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
    model = YOLO('yolo11n-cls.pt')
    model.train(
        data=str(DATA_ROOT), epochs=MAX_EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE,
        device=0, workers=2, optimizer='AdamW', lr0=1e-3, weight_decay=5e-4,
        patience=EARLY_STOP_PATIENCE, seed=SEED, deterministic=True, split='val',
        val=True, save=True, plots=True, project=str(YOLO_RUN_DIR.parent),
        name=YOLO_RUN_DIR.name, exist_ok=False, verbose=True,
    )
    BEST_CHECKPOINT = YOLO_RUN_DIR / 'weights' / 'best.pt'
    assert BEST_CHECKPOINT.is_file(), f'Best checkpoint was not saved: {BEST_CHECKPOINT}'
    environment = {
        'model': 'yolo11n-cls.pt', 'seed': SEED, 'imgsz': IMG_SIZE,
        'epochs_max': MAX_EPOCHS, 'patience': EARLY_STOP_PATIENCE,
        'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
        'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
        'train_manifest_sha256': sha256_file(TRAIN_CSV),
        'val_manifest_sha256': sha256_file(VAL_CSV), 'test_used_during_training': False,
    }
    (YOLO_RUN_DIR / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
    print('Best checkpoint:', BEST_CHECKPOINT)
else:
    print('Training not started. Review preflight, then set START_TRAINING=True.')

## Loss curves and model footprint

After training, Ultralytics saves results.csv and plots. This cell displays train/validation loss and records parameters and best-checkpoint size. It does not load test images.

In [ ]:
if YOLO_RUN_DIR.is_dir() and (YOLO_RUN_DIR / 'results.csv').is_file():
    import matplotlib.pyplot as plt
    results = pd.read_csv(YOLO_RUN_DIR / 'results.csv')
    results.columns = [c.strip() for c in results.columns]
    train_col = next((c for c in results.columns if c.lower() == 'train/loss'), None)
    val_col = next((c for c in results.columns if c.lower() == 'val/loss'), None)
    if train_col and val_col:
        x = results['epoch'] if 'epoch' in results else np.arange(1, len(results) + 1)
        plt.figure(figsize=(8, 5))
        plt.plot(x, results[train_col], label='Train')
        plt.plot(x, results[val_col], label='Validation')
        plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('YOLO11n-cls loss curves')
        plt.grid(True); plt.legend(); plt.show()
    else:
        print('Loss columns not found:', results.columns.tolist())
    best_path = YOLO_RUN_DIR / 'weights' / 'best.pt'
    if best_path.is_file():
        footprint_model = YOLO(str(best_path))
        params = sum(p.numel() for p in footprint_model.model.parameters())
        print({'parameters': params, 'best_checkpoint_bytes': best_path.stat().st_size,
               'best_checkpoint_MiB': round(best_path.stat().st_size / (1024 ** 2), 3)})
else:
    print('Run the training cell first to create results.csv and best.pt.')

## LiteRT and Core ML exports (explicit opt-in)

Exports use the validation-selected best.pt and do not use test data. Ultralytics documents the litert and coreml formats for classification. If Core ML conversion is unsupported in the active Colab image, export on a supported Linux/macOS runtime using the same checkpoint. [Export guide](https://docs.ultralytics.com/modes/export).

In [ ]:
if RUN_EXPORTS:
    best_path = YOLO_RUN_DIR / 'weights' / 'best.pt'
    assert best_path.is_file(), f'Train first; checkpoint not found: {best_path}'
    export_model = YOLO(str(best_path))
    litert_path = export_model.export(format='litert', imgsz=IMG_SIZE)
    print('LiteRT export:', litert_path)
    coreml_path = export_model.export(format='coreml', imgsz=IMG_SIZE)
    print('Core ML export:', coreml_path)
else:
    print('Exports not started. Set RUN_EXPORTS=True after best.pt exists.')

## Outputs

Drive run folder: runs/rice_classification/yolo11n_cls/. It contains best.pt, last.pt, results.csv, Ultralytics plots, and environment.json. Optional exports run only when RUN_EXPORTS is enabled. No accuracy result is prefilled.